# Correct camera-vs-stage rotation

Every microscope's camera sensor sits at some small, fixed angle relative to
the stage's true X/Y travel axes. A FOV grid generated by
`acquisition.positions.create_grid_positions` assumes perfect alignment
between the two -- unrealistic in practice, and different microscopes carry
different amounts of this rotation. Left uncorrected, adjacent FOVs'
overlapping borders don't actually line up where the grid assumes they do,
which can lose or duplicate cells/barcodes at FOV borders during MERlin
segmentation/decoding.

**Method** (validated manually with BigStitcher before this notebook
automated it -- see `notes_BC341.pdf` and the `230818_BC341_adding_camera_
angle_correction_to_merlin` folder, both in
`BreastCancer/notebook/`): sample a handful of anchor FOVs (default 10),
register each one against its real 4-connected neighbours in their expected
overlapping border (phase cross-correlation on a DAPI/tissue-channel frame
from an already-imaged **cells round**), and fit ONE affine transform from
every anchor+neighbour correspondence pooled together
(`MERci.acquisition.camera_rotation`, using the `affine6p` package -- the
same one validated in the original BigStitcher-replacement work). Since the
rotation is a fixed optical-path property, this single global transform
corrects every FOV in the experiment, not just the sampled ones.

**Scope**: single-positions-file (legacy/single-tissue) layout only, matching
`measure_tissue_thickness_test.ipynb`/`stage_drift_dapi.ipynb`'s own
assumption -- the multi-boundary per-segment `positions_file` layout is not
handled here.

**Outputs**:
- `positions/camera_rotation_correction_{POSITIONS_TAG}.npy` -- the fitted
  3x3 affine matrix (`MERci.acquisition.camera_rotation.
  CameraRotationCorrection`, `affine6p` convention)
- `positions/camera_rotation_correction_{POSITIONS_TAG}.json` -- fit metadata
  (rotation/scale, correspondence count, source round)
- `positions/positions_{POSITIONS_TAG}_corrected.txt` -- the corrected FOV
  positions. **Use this file as MERlin's positions input instead of the raw
  grid** -- `07_create_merlin_scripts.ipynb` auto-detects it (falls back to
  the raw file if this notebook hasn't been run).


## 1 — Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import KDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.io              import load_positions, save_positions_array, read_image_frames
from MERci.common.experiment_info import load_experiment_info, resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs    import (
    find_frame_table_for_hal_config, get_camera_pixel_size_um, get_camera_frame_size,
)
from MERci.acquisition.positions  import find_grid_neighbor
from MERci.acquisition.camera_rotation import (
    NeighborCorrespondence, sample_neighbor_correspondences, fit_camera_rotation,
)
from MERci.scheduler        import resolve_round_flip_y
from MERci.analysis.round   import load_raw_frames_for_round, create_mosaic_ffc
from MERci.progress_display import ProgressReporter

NOTEBOOK_NAME = "correct_camera_rotation"

print(f"SAMPLE_DIR : {SAMPLE_DIR}")


## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Read from experiment_info.yaml (notebook 06) rather than hard-coding it --
# get_camera_pixel_size_um/get_camera_frame_size below are keyed on the EXACT
# microscope string, and a wrong guess here silently produces a wrong pixel
# size/frame geometry with no error (confirmed directly: a hard-coded wrong
# microscope in an earlier version of this notebook produced phase-
# correlation registrations that all failed silently -- error=1.0 on every
# correspondence -- because the assumed overlap-border crop no longer matched
# where the real images actually overlap).
info       = load_experiment_info(SAMPLE_DIR / "metadata" / "experiment_info.yaml")
MICROSCOPE = info.microscope

# Which already-imaged round to measure the rotation from -- by imaging_type
# (typically "cells": every FOV is imaged there, with a strong DAPI/tissue
# signal to register on), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Path to a specific frame-table CSV, bypassing auto-resolution from the
# round's HAL config's <shutters> reference (find_frame_table_for_hal_config)
# -- set this when metadata/ has drifted out of sync with the real frame
# table used at acquisition time. None (default) auto-resolves as usual.
REFERENCE_FRAME_TABLE_PATH = None

# Registration colour (nm) and z (um) -- same convention as stage_drift_dapi.ipynb.
# No sensible default exists across experiments; this MUST be set explicitly to
# a z where REGISTRATION_COLOR_NM shows real tissue signal (e.g. from
# measure_tissue_thickness_test.ipynb's z_first_um/z_last_um for this round).
REGISTRATION_COLOR_NM = 405.0
REGISTRATION_Z_UM     = None

# How many anchor FOVs to sample and register against their 4-connected
# neighbours (up to 4 correspondences each). The fit pools every anchor's
# correspondences into ONE affine (camera_rotation.fit_camera_rotation) --
# 10 anchors x up to 4 neighbours each is normally far more than the 3 points
# a 2-D affine strictly needs, giving the fit real redundancy against any one
# bad registration.
N_ANCHORS = 10

# Deterministic anchor sampling by default (reproducible re-runs); None for a
# fresh random sample every run.
SEED = 0

# Match tolerance for "is there a real FOV at this grid-neighbour position",
# as a fraction of step_size_um -- same default as acquisition.positions.
# find_exterior_fovs / find_grid_neighbor.
TOLERANCE_FRACTION = 0.25

# Sub-pixel registration precision (1/upsample_factor px); 10 -> 0.1 px, the
# same default acquisition.alignment.compute_fov_drifts uses.
UPSAMPLE_FACTOR = 10

# Zero the fitted affine's translation component -- see camera_rotation.
# fit_camera_rotation's docstring: every correspondence only measures a
# LOCAL, anchor-relative displacement, so a non-zero fitted translation
# reflects pooling noise across scattered anchors, not a real global origin
# shift.
ZERO_TRANSLATION = True

# Skip re-registering if a cached correspondences table already exists (see
# NOTEBOOK_GUIDELINES.md #3). Set True to force a fresh measurement (e.g.
# after changing N_ANCHORS/SEED/REGISTRATION_Z_UM).
FORCE_RECOMPUTE = False

# Half-width (um) of the zoomed corner-overlay crop (plot 4) -- small enough
# to show pixel-scale misalignment, large enough to still contain real
# tissue signal. Tune per-experiment if the default crop looks empty/noisy.
CORNER_ZOOM_HALF_UM = 15.0

# Mosaic-comparison rendering (plot 3) -- deliberately NOT cropping each
# FOV's overlap border (crop_px=0), unlike the production FFC mosaics in
# analysis/round.py: the whole point of this comparison is to SEE the
# overlap region, where camera rotation shows up as doubled/discontinuous
# tissue edges before correction.
MOSAIC_THUMBNAIL_SIZE = (200, 200)
MOSAIC_CROP_PX         = 0

# The mosaic comparison (plot 3) reads every included FOV's raw frame
# individually -- for a real multi-thousand-FOV round over a network drive,
# reading ALL of them (as a first version of this notebook did) took over 30
# minutes and never finished. Restrict it to a contiguous
# MOSAIC_WINDOW_N_FOVS x MOSAIC_WINDOW_N_FOVS block of FOVs around the grid's
# centre instead -- still enough real 4-connected overlap borders to see
# whether the correction reduced the doubled/discontinuous seams, at a
# fraction of the read time. Raise it if the block looks too small once you
# see the plot.
MOSAIC_WINDOW_N_FOVS = 10

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5).
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Sample name   : {SAMPLE_NAME}")
print(f"Positions tag : {POSITIONS_TAG}")
print(f"Microscope    : {MICROSCOPE}")
print(f"N_ANCHORS     : {N_ANCHORS}")


## 3 — Resolve the reference round + registration frame

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

print("Available rounds (data folders):")
for rid in meta.valid_round_ids():
    series_list = meta.series_for_round(rid)
    types = sorted({(s.imaging_type or "?") for s in series_list})
    print(f"  round {rid:>2}  imaging_type={types}")


def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta, frame_table_path=None):
    """(frame_table, hal_config_path, SeriesInfo) for round_id's first series with a
    hal_config. Mirrors stage_drift_dapi.ipynb's own helper of the same name."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        if frame_table_path is not None:
            return pd.read_csv(frame_table_path, index_col=0), hal_path, s
        ft_path = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0), hal_path, s
    raise FileNotFoundError(f"No frame table found for round {round_id}")


def nearest_frame_for_color_z(frame_table, color_nm, z_um):
    """Frame index of the given colour's frame with z closest to z_um."""
    candidates = frame_table[frame_table["color"].round(0) == round(color_nm)]
    if candidates.empty:
        raise ValueError(f"No frames of colour {color_nm:.0f} nm in this frame table.")
    return int((candidates["z"] - z_um).abs().idxmin())


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"some FOVs sampled below may be missing.")

if REGISTRATION_Z_UM is None:
    raise ValueError(
        "REGISTRATION_Z_UM is not set (section 2) -- pick a z (um) where "
        f"REGISTRATION_COLOR_NM ({REGISTRATION_COLOR_NM:.0f} nm) actually shows real "
        "signal (e.g. from measure_tissue_thickness_test.ipynb's z_first_um/z_last_um) "
        "before continuing."
    )

frame_table, reference_hal_config_path, reference_series = load_round_frame_table(
    target_round_id, config, meta, frame_table_path=REFERENCE_FRAME_TABLE_PATH,
)
reference_frame_idx = nearest_frame_for_color_z(frame_table, REGISTRATION_COLOR_NM, REGISTRATION_Z_UM)
reference_frame_z   = float(frame_table.loc[reference_frame_idx, "z"])

PIXEL_SIZE_UM     = get_camera_pixel_size_um(MICROSCOPE)
FRAME_WIDTH_PX, _ = get_camera_frame_size(MICROSCOPE)
FRAME_WIDTH_UM    = FRAME_WIDTH_PX * PIXEL_SIZE_UM
FLIP_Y            = resolve_round_flip_y(target_round_id, config, meta)

full_positions = load_positions(config.positions_txt)   # {fov_id: (x, y)} -- nominal grid
fov_order = sorted(full_positions)

# STEP_SIZE_UM/OVERLAP_FRACTION are measured from the REAL positions file
# rather than trusted from ExperimentConfig's own step_size_um/
# non_overlap_fraction formula -- confirmed directly that trusting the
# formula (which depends on config.pixel_size_um/image_size_px/
# non_overlap_fraction matching this experiment's real camera/grid exactly)
# silently produced a wrong overlap-border crop when those didn't match,
# and every phase-correlation registration failed (error=1.0) as a result.
# The median nearest-neighbour distance across real FOV positions is the
# grid's true step size regardless of what generated it.
coords_arr       = np.array([full_positions[f] for f in fov_order])
nn_dist, _       = KDTree(coords_arr).query(coords_arr, k=2)
STEP_SIZE_UM     = float(np.median(nn_dist[:, 1]))
OVERLAP_FRACTION = max(0.0, 1.0 - STEP_SIZE_UM / FRAME_WIDTH_UM)

print(f"Target round      : {target_round_id}  (series pattern: {reference_series.name!r})")
print(f"Reference frame    : index {reference_frame_idx}, colour {REGISTRATION_COLOR_NM:.0f} nm, "
      f"z={reference_frame_z:.2f} um (requested z={REGISTRATION_Z_UM:.2f} um)")
print(f"Pixel size (um)    : {PIXEL_SIZE_UM}")
print(f"Frame width (um)   : {FRAME_WIDTH_UM:.3f}  ({FRAME_WIDTH_PX} px)")
print(f"Step size (um)     : {STEP_SIZE_UM:.3f}  (measured, median nearest-neighbour distance)")
print(f"Overlap fraction   : {OVERLAP_FRACTION:.3f}")
print(f"flip_y             : {FLIP_Y}")
print(f"FOVs in this round : {len(full_positions)}")


## 4 — Sample neighbour correspondences (calculation, cached)

Registers `N_ANCHORS` randomly-sampled FOVs against their real 4-connected
neighbours in the DAPI/tissue channel (phase cross-correlation on the
overlapping border crop). Cached to
`analysis/cache/correct_camera_rotation/neighbor_correspondences.csv` so
re-running the notebook doesn't re-register unless `FORCE_RECOMPUTE=True`.


In [ ]:
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
correspondences_csv = CACHE_DIR / "neighbor_correspondences.csv"


def load_dapi_frame(fov_id):
    path = reference_series.resolve_path(fov_id, config.image_suffix)
    return read_image_frames(path, [reference_frame_idx],
                              frame_width=config.frame_width, frame_height=config.frame_height)[0]


if correspondences_csv.exists() and not FORCE_RECOMPUTE:
    correspondences_df = pd.read_csv(correspondences_csv)
    print(f"Loaded cached correspondences: {correspondences_csv}  ({len(correspondences_df)} rows)")
else:
    n_anchors_actual = min(N_ANCHORS, len(fov_order))
    reporter = ProgressReporter(total=n_anchors_actual * 4, label="Registering neighbour pairs")

    def _progress(done, total):
        reporter.update(done - reporter.n_done)

    correspondences = sample_neighbor_correspondences(
        fov_ids=fov_order,
        positions=full_positions,
        load_frame=load_dapi_frame,
        step_size_um=STEP_SIZE_UM,
        pixel_size_um=PIXEL_SIZE_UM,
        overlap_fraction=OVERLAP_FRACTION,
        n_anchors=N_ANCHORS,
        tolerance_fraction=TOLERANCE_FRACTION,
        upsample_factor=UPSAMPLE_FACTOR,
        seed=SEED,
        progress_callback=_progress,
    )
    reporter.done()

    correspondences_df = pd.DataFrame([{
        "anchor_fov": c.anchor_fov, "neighbor_fov": c.neighbor_fov, "direction": c.direction,
        "nominal_x": c.nominal_xy[0], "nominal_y": c.nominal_xy[1],
        "measured_x": c.measured_xy[0], "measured_y": c.measured_xy[1],
        "error": c.error,
    } for c in correspondences])
    correspondences_df.to_csv(correspondences_csv, index=False)
    print(f"Sampled {len(correspondences_df)} neighbour correspondence(s) from "
          f"{n_anchors_actual} anchor FOV(s). Saved: {correspondences_csv}")

print(correspondences_df.head())


## 5 — Fit the camera-rotation affine transform

In [ ]:
correspondences = [
    NeighborCorrespondence(
        anchor_fov=int(r.anchor_fov), neighbor_fov=int(r.neighbor_fov), direction=r.direction,
        nominal_xy=(r.nominal_x, r.nominal_y), measured_xy=(r.measured_x, r.measured_y), error=r.error,
    )
    for r in correspondences_df.itertuples()
]

correction = fit_camera_rotation(correspondences, zero_translation=ZERO_TRANSLATION)

M = correction.matrix
rotation_deg = float(np.degrees(np.arctan2(M[1, 0], M[0, 0])))
scale        = float(np.sqrt(M[0, 0] ** 2 + M[1, 0] ** 2))
print(f"Fitted from {correction.n_correspondences} correspondence(s):")
print(f"  rotation : {rotation_deg:.4f} deg")
print(f"  scale    : {scale:.6f}")
print(f"  matrix   :\n{M}")

positions_dir = SAMPLE_DIR / "positions"
matrix_path = positions_dir / f"camera_rotation_correction_{POSITIONS_TAG}.npy"
correction.save(matrix_path)

meta_path = positions_dir / f"camera_rotation_correction_{POSITIONS_TAG}.json"
with meta_path.open("w") as fh:
    json.dump({
        "n_correspondences": correction.n_correspondences,
        "zero_translation": correction.zero_translation,
        "rotation_deg": rotation_deg,
        "scale": scale,
        "source_round_id": target_round_id,
        "registration_color_nm": REGISTRATION_COLOR_NM,
        "registration_z_um": REGISTRATION_Z_UM,
    }, fh, indent=2)

print(f"Saved correction matrix  : {matrix_path}")
print(f"Saved correction metadata: {meta_path}")


## 6 — Apply the correction to the full positions file

In [ ]:
nominal_coords   = np.array([full_positions[f] for f in fov_order], dtype=float)
corrected_coords = correction.transform_points(nominal_coords)
corrected_positions = {f: tuple(corrected_coords[i]) for i, f in enumerate(fov_order)}

corrected_path = positions_dir / f"positions_{POSITIONS_TAG}_corrected.txt"
save_positions_array(corrected_coords, corrected_path)

print(f"Applied camera-rotation correction to all {len(full_positions)} FOV(s).")
print(f"Wrote corrected positions file: {corrected_path}")
print("Use this file as MERlin's positions input instead of the raw grid "
      "(07_create_merlin_scripts.ipynb auto-detects it).")

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


## 7 — Plot: original vs. corrected positions overlay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(nominal_coords[:, 0], nominal_coords[:, 1], "o-", ms=3, lw=0.5, color="tab:blue", label="original")
ax.plot(corrected_coords[:, 0], corrected_coords[:, 1], "o-", ms=3, lw=0.5, color="tab:red", label="corrected")
ax.invert_yaxis()
ax.axis("equal")
ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Original vs. corrected FOV positions -- {SAMPLE_NAME}\n"
             f"rotation={rotation_deg:.3f} deg, scale={scale:.5f}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()

fig_path = figures_dir / f"{NOTEBOOK_NAME}.positions_overlay.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 8 — Plot: original & corrected positions with FOV grids

In [ ]:
fov_size_um = FRAME_WIDTH_UM
half = fov_size_um / 2

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, coords, title in ((axes[0], nominal_coords, "original"), (axes[1], corrected_coords, "corrected")):
    for x, y in coords:
        ax.add_patch(mpatches.Rectangle((x - half, y - half), fov_size_um, fov_size_um,
                                         lw=0.3, edgecolor="tab:blue", facecolor="tab:blue", alpha=0.15))
    ax.plot(coords[:, 0], coords[:, 1], "o", ms=2, color="k")
    ax.invert_yaxis()
    ax.axis("equal")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"FOV footprints before/after camera-rotation correction -- {SAMPLE_NAME}",
             fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.fov_grids.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 9 — Plot: original vs. corrected DAPI mosaic comparison

Reuses the SAME raw DAPI frames for both mosaics -- only the tile
*placement* differs (original vs. corrected positions) -- via
`analysis.round.create_mosaic_ffc`. Deliberately uncropped (`MOSAIC_CROP_PX
= 0`, section 2) so the overlap region is visible: camera rotation shows up
as doubled/discontinuous tissue edges there before correction.

Restricted to a `MOSAIC_WINDOW_N_FOVS`-wide contiguous block of FOVs around
the grid centre (section 2) -- reading every FOV in a real multi-thousand-FOV
round over a network drive is a 30+ minute read for a single diagnostic
plot; a local block still shows real 4-connected overlap borders.


In [ ]:
center_xy      = np.median(np.array(list(full_positions.values())), axis=0)
half_window_um = MOSAIC_WINDOW_N_FOVS * STEP_SIZE_UM / 2
window_fov_ids = [
    f for f, (x, y) in full_positions.items()
    if abs(x - center_xy[0]) <= half_window_um and abs(y - center_xy[1]) <= half_window_um
]
print(f"Mosaic-comparison window: {len(window_fov_ids)} FOV(s) around "
      f"({center_xy[0]:.1f}, {center_xy[1]:.1f}) um "
      f"(+/-{half_window_um:.1f} um -- MOSAIC_WINDOW_N_FOVS={MOSAIC_WINDOW_N_FOVS}).")

raw_frames, _ = load_raw_frames_for_round(
    target_round_id, meta, reference_frame_idx, fov_subset=window_fov_ids,
    frame_width=config.frame_width, frame_height=config.frame_height,
)
print(f"Loaded {len(raw_frames)} raw DAPI frame(s) for round {target_round_id}.")


In [ ]:
original_mosaic_path  = CACHE_DIR / f"mosaic_round{target_round_id}_original.png"
corrected_mosaic_path = CACHE_DIR / f"mosaic_round{target_round_id}_corrected.png"

original_positions_subset  = {f: full_positions[f] for f in raw_frames}
corrected_positions_subset = {f: corrected_positions[f] for f in raw_frames}

original_canvas = create_mosaic_ffc(
    raw_frames, original_positions_subset, original_mosaic_path,
    crop_px=MOSAIC_CROP_PX, thumbnail_size=MOSAIC_THUMBNAIL_SIZE, flip_y=FLIP_Y,
)
corrected_canvas = create_mosaic_ffc(
    raw_frames, corrected_positions_subset, corrected_mosaic_path,
    crop_px=MOSAIC_CROP_PX, thumbnail_size=MOSAIC_THUMBNAIL_SIZE, flip_y=FLIP_Y,
)
print(f"Saved: {original_mosaic_path}")
print(f"Saved: {corrected_mosaic_path}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, canvas, title in ((axes[0], original_canvas, "original"), (axes[1], corrected_canvas, "corrected")):
    ax.imshow(canvas, cmap="gray")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
fig.suptitle(f"DAPI round mosaic, before vs. after camera-rotation correction -- round {target_round_id}",
             fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.mosaic_comparison.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 10 — Plot: corner overlay of a 2x2 (8-connected) FOV block

Finds one complete 2x2 block (anchor + its right/up/diagonal neighbours),
places their raw DAPI frames at their real stage positions (nominal, then
corrected), each tinted a distinct colour and combined by per-channel max,
then zooms into the shared corner. A doubled/coloured-fringe seam means the
tiles' real content doesn't actually line up where that position set assumes
it does -- the same visual check as the notes' BigStitcher verification
plots, extended from 2 to 4 tiles.


In [ ]:
def find_corner_block(fov_ids, positions, step_size_um, tolerance_fraction):
    """First (anchor, right, up, diag) 2x2 (8-connected) block found among
    fov_ids where all four FOVs are real."""
    for anchor in fov_ids:
        right = find_grid_neighbor(anchor, positions, "right", step_size_um, tolerance_fraction)
        up    = find_grid_neighbor(anchor, positions, "up",    step_size_um, tolerance_fraction)
        if right is None or up is None:
            continue
        diag = find_grid_neighbor(right, positions, "up", step_size_um, tolerance_fraction)
        if diag is None:
            diag = find_grid_neighbor(up, positions, "right", step_size_um, tolerance_fraction)
        if diag is not None:
            return anchor, right, up, diag
    return None


block = find_corner_block(fov_order, full_positions, STEP_SIZE_UM, TOLERANCE_FRACTION)
if block is None:
    raise RuntimeError("No complete 2x2 (8-connected) FOV block found -- "
                        "try a larger TOLERANCE_FRACTION.")
anchor_fov, right_fov, up_fov, diag_fov = block
block_names = {"anchor": anchor_fov, "right": right_fov, "up": up_fov, "diag": diag_fov}
print(f"Corner block: {block_names}")

block_frames = {name: load_dapi_frame(fov_id) for name, fov_id in block_names.items()}


def build_corner_canvas(frames, positions_um, pixel_size_um, colors, flip_y):
    """Place full raw frames at their real stage positions (nominal or
    corrected), each tinted a distinct colour, combined by per-channel max --
    a doubled/coloured-fringe seam means the two tiles' real content doesn't
    actually line up where the given position set assumes it does."""
    h, w = next(iter(frames.values())).shape
    ys_sign = -1.0 if flip_y else 1.0
    xs = [positions_um[n][0] for n in frames]
    ys = [ys_sign * positions_um[n][1] for n in frames]
    x_min = min(xs) - w * pixel_size_um / 2
    y_min = min(ys) - h * pixel_size_um / 2
    x_max = max(xs) + w * pixel_size_um / 2
    y_max = max(ys) + h * pixel_size_um / 2
    canvas_w = int(round((x_max - x_min) / pixel_size_um))
    canvas_h = int(round((y_max - y_min) / pixel_size_um))
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.float32)
    for name, img in frames.items():
        lo, hi = np.percentile(img, [1, 99.5])
        norm = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1.0), 0, 1)
        x_um, y_um = positions_um[name][0], ys_sign * positions_um[name][1]
        col0 = int(round((x_um - w * pixel_size_um / 2 - x_min) / pixel_size_um))
        row0 = int(round((y_um - h * pixel_size_um / 2 - y_min) / pixel_size_um))
        tile_rgb = norm[..., None] * np.array(colors[name], dtype=np.float32)
        canvas[row0:row0 + h, col0:col0 + w] = np.maximum(canvas[row0:row0 + h, col0:col0 + w], tile_rgb)
    return canvas, (x_min, y_min)


BLOCK_COLORS = {"anchor": (1.0, 0.0, 0.0), "right": (0.0, 1.0, 0.0),
                "up": (0.0, 0.0, 1.0), "diag": (1.0, 1.0, 0.0)}

original_block_positions  = {name: full_positions[fov_id] for name, fov_id in block_names.items()}
corrected_block_positions = {name: corrected_positions[fov_id] for name, fov_id in block_names.items()}

canvas_before, _ = build_corner_canvas(block_frames, original_block_positions, PIXEL_SIZE_UM, BLOCK_COLORS, FLIP_Y)
canvas_after, _  = build_corner_canvas(block_frames, corrected_block_positions, PIXEL_SIZE_UM, BLOCK_COLORS, FLIP_Y)

print(f"Corner canvases built: before={canvas_before.shape[:2]}, after={canvas_after.shape[:2]}")


In [ ]:
def crop_center(canvas, half_um, pixel_size_um):
    h, w = canvas.shape[:2]
    half_px = int(round(half_um / pixel_size_um))
    cy, cx = h // 2, w // 2
    r0, r1 = max(cy - half_px, 0), min(cy + half_px, h)
    c0, c1 = max(cx - half_px, 0), min(cx + half_px, w)
    return canvas[r0:r1, c0:c1]


crop_before = crop_center(canvas_before, CORNER_ZOOM_HALF_UM, PIXEL_SIZE_UM)
crop_after  = crop_center(canvas_after,  CORNER_ZOOM_HALF_UM, PIXEL_SIZE_UM)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, crop, title in ((axes[0], crop_before, "before correction"), (axes[1], crop_after, "after correction")):
    ax.imshow(crop)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
handles = [mpatches.Patch(color=c, label=n) for n, c in BLOCK_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=PLOT_LEGEND_FONTSIZE)
fig.suptitle(f"Corner overlay of FOVs {list(block_names.values())} (8-connected) -- "
             f"before vs. after correction", fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig_path = figures_dir / f"{NOTEBOOK_NAME}.corner_overlay.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")
